In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("../Raw-data/Nigerian_Fraud.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (3332, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']


,sender,receiver,date,subject,body,urls,label
0,MR. JAMES NGOLA. <james_ngola2002@maktoob.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 02:38:20 +0000",URGENT BUSINESS ASSISTANCE AND PARTNERSHIP,FROM:MR. JAMES NGOLA.\nCONFIDENTIAL TEL: 233-2...,0,1
1,Mr. Ben Suleman <bensul2004nng@spinfinder.com>,R@M,"Thu, 31 Oct 2002 05:10:00 -0000",URGENT ASSISTANCE /RELATIONSHIP (P),"Dear Friend,\n\nI am Mr. Ben Suleman a custom ...",0,1
2,PRINCE OBONG ELEME <obong_715@epatra.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 22:17:55 +0100",GOOD DAY TO YOU,FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,0,1
3,PRINCE OBONG ELEME <obong_715@epatra.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 22:44:20 -0000",GOOD DAY TO YOU,FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,0,1
4,Maryam Abacha <m_abacha03@www.com>,R@M,"Fri, 01 Nov 2002 01:45:04 +0100",I Need Your Assistance.,"Dear sir, \n \nIt is with a heart full of hope...",0,1


In [3]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["subject_clean"] = df["subject"].apply(clean_text)
df["body_clean"] = df["body"].apply(clean_text)

df["email_text"] = (
    df["subject_clean"] + " " + df["body_clean"]
).str.strip()

df["label"] = pd.to_numeric(df["label"], errors="coerce")

In [5]:
clean_df = df[["email_text", "label"]].copy()

clean_df = clean_df.dropna(subset=["email_text", "label"])
clean_df = clean_df[clean_df["email_text"].str.len() > 0]
clean_df = clean_df.drop_duplicates(subset=["email_text"]).reset_index(drop=True)

clean_df["label"] = clean_df["label"].astype(int)

print("Cleaned shape:", clean_df.shape)
print(clean_df["label"].value_counts())
clean_df.head()

Cleaned shape: (3293, 2)
label
1    3293
Name: count, dtype: int64


,email_text,label
0,URGENT BUSINESS ASSISTANCE AND PARTNERSHIP FRO...,1
1,URGENT ASSISTANCE /RELATIONSHIP (P) Dear Frien...,1
2,GOOD DAY TO YOU FROM HIS ROYAL MAJESTY (HRM) C...,1
3,GOOD DAY TO YOU FROM HIS ROYAL MAJESTY (HRM) C...,1
4,"I Need Your Assistance. Dear sir, It is with a...",1


In [6]:
clean_df["email_text"] = clean_df["email_text"].astype(str)

clean_df["email_text"].str.contains(
    r"\b(?:spam|ham|phishing|legitimate)\b",
    case=False,
    na=False
).mean()

np.float64(0.09231703613726086)

In [7]:
clean_df.to_csv("../Processed-Data/nigerian_fraud_cleaned.csv", index=False)
print("Nigerian Fraud dataset cleaned and saved!")

Nigerian Fraud dataset cleaned and saved!
